In [19]:
import os
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

In [21]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.models as models
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np

# 1. إعدادات الجهاز
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"💻 Using device: {device}")

# 2. تجميع كل مسارات الصور والـ Labels من الداتا الأصلية
all_images = []
all_labels = []
data_root = r"D:\__Projects\Graduation---Project\DL\Brain_Stroke\Data" 
class_mapping = {'Normal': 0, 'Ischemia': 1, 'Bleeding': 2}

for folder_name, class_idx in class_mapping.items():
    png_path = os.path.join(data_root, folder_name, 'PNG')
    if os.path.exists(png_path):
        for img_name in os.listdir(png_path):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                all_images.append(os.path.join(png_path, img_name))
                all_labels.append(class_idx)

print(f" Total Dataset Size Found: {len(all_images)} slices.")

# =========================================================================
# 3. التقسيم الاحترافي الصارم (Stratified Split) لمنع الـ Data Leakage نهائياً
# =========================================================================
# الخطوة أ: عزل 15% كاملة ومستقلة للـ Test Set النهائي
train_val_imgs, test_imgs, train_val_lbls, test_lbls = train_test_split(
    all_images, all_labels, test_size=0.15, stratify=all_labels, random_state=42
)

# الخطوة ب: تقسيم الـ 85% المتبقية لـ Train (85%) و Validation (15%)
train_imgs, val_imgs, train_lbls, val_lbls = train_test_split(
    train_val_imgs, train_val_lbls, test_size=0.15, stratify=train_val_lbls, random_state=42
)

print(f" Data Splits: Train={len(train_imgs)} | Val={len(val_imgs)} | Test={len(test_imgs)}")

# 4. بناء الـ PyTorch Dataset
class BrainStrokeDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
        
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('L') # قراءة Grayscale للـ CT
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, label

# 5. الـ Data Augmentation والـ Transforms
# الـ ResNet18 متوقعة 3 قنوات، فعملنا Repeat للقناة الواحدة لتوافق الـ Weights الجاهزة
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)), 
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# إنشاء الـ Loaders
train_loader = DataLoader(BrainStrokeDataset(train_imgs, train_lbls, train_transform), batch_size=32, shuffle=True)
val_loader = DataLoader(BrainStrokeDataset(val_imgs, val_lbls, val_test_transform), batch_size=32, shuffle=False)
test_loader = DataLoader(BrainStrokeDataset(test_imgs, test_lbls, val_test_transform), batch_size=32, shuffle=False)

# 6. تعريف وتحضير موديل ResNet18 من جديد
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 3) # 3 كلاسات: Normal, Ischemia, Bleeding
model = model.to(device)

# حساب الـ Class Weights لعلاج الـ Imbalance لو الـ Ischemia أو Bleeding قليلين
class_counts = np.bincount(train_lbls)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()
weight_tensor = torch.FloatTensor(class_weights).to(device)

criterion = nn.CrossEntropyLoss(weight=weight_tensor)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

# 7. حلقة التدريب (Training Loop)
epochs = 10  # غير الرقم بناءً على وقتك وحجم الداتا
best_val_acc = 0.0

print("\n Starting Training Loop...")
for epoch in range(epochs):
    model.train()
    train_loss, train_correct = 0.0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        train_correct += torch.sum(preds == labels.data)
        
    train_loss = train_loss / len(train_loader.dataset)
    train_acc = train_correct.double() / len(train_loader.dataset)
    
    # الـ Validation
    model.eval()
    val_loss, val_correct = 0.0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            val_correct += torch.sum(preds == labels.data)
            
    val_loss = val_loss / len(val_loader.dataset)
    val_acc = val_correct.double() / len(val_loader.dataset)
    
    print(f"Epoch {epoch+1}/{epochs} -> Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
    
    # حفظ أفضل أوزان بناءً على أداء الـ Validation
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), r'D:\__Projects\Graduation---Project\app\models\stroke_resnet18.pth')

print("\n Training complete. Best Validation Accuracy weights saved.")

# =========================================================================
# 8. التقييم النهائي الصارم على الـ Test Set الحقيقي والنظيف
# =========================================================================
print("\n Evaluating Best Model on the Untouched Test Set...")
model.load_state_dict(torch.load(r'D:\__Projects\Graduation---Project\app\models\stroke_resnet18.pth'))
model.eval()

all_preds = []
all_test_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_test_labels.extend(labels.numpy())

print("\n Final Classification Report (Pure Test Set - No Leakage):")
print(classification_report(all_test_labels, all_preds, target_names=['Normal', 'Ischemia', 'Bleeding']))

💻 Using device: cuda
 Total Dataset Size Found: 6650 slices.
 Data Splits: Train=4804 | Val=848 | Test=998

 Starting Training Loop...
Epoch 1/10 -> Train Loss: 0.5917 Acc: 0.7644 | Val Loss: 0.4311 Acc: 0.7700
Epoch 2/10 -> Train Loss: 0.3380 Acc: 0.8695 | Val Loss: 0.2930 Acc: 0.8833
Epoch 3/10 -> Train Loss: 0.2660 Acc: 0.9070 | Val Loss: 0.2811 Acc: 0.8915
Epoch 4/10 -> Train Loss: 0.1570 Acc: 0.9365 | Val Loss: 0.2473 Acc: 0.9363
Epoch 5/10 -> Train Loss: 0.1645 Acc: 0.9369 | Val Loss: 0.2274 Acc: 0.8903
Epoch 6/10 -> Train Loss: 0.1106 Acc: 0.9611 | Val Loss: 0.2716 Acc: 0.9045
Epoch 7/10 -> Train Loss: 0.0819 Acc: 0.9700 | Val Loss: 0.2426 Acc: 0.9316
Epoch 8/10 -> Train Loss: 0.0759 Acc: 0.9684 | Val Loss: 0.2369 Acc: 0.9340
Epoch 9/10 -> Train Loss: 0.1036 Acc: 0.9559 | Val Loss: 0.1756 Acc: 0.9363
Epoch 10/10 -> Train Loss: 0.1154 Acc: 0.9542 | Val Loss: 0.2061 Acc: 0.9587

 Training complete. Best Validation Accuracy weights saved.

 Evaluating Best Model on the Untouched Te